# 04 — Telperion: The Complete Un-Extinctable Bulk

**The Zero Tree / Telperion — Notebook 4 of 4**

---

This notebook assembles the full Telperion dataset and produces the data  
that drives all three Blender renders.

**Telperion is:**
- The union of all prime paths from k=0 (ℝ leaves) to k=8 (T_256 root)
- Stratified by N-shape: 16 strata, indexed by sedenion basis element
- Silver leaves: Monster gap primes (p ≡ 1, 11, 15 mod 16) — no Niemeier structure reaches them
- Complete: every prime leaf has a path; no prime path is broken by a ZD crossing

**The N-shape theorem** (proved in FermatMonster engine v0.300):  
The 71 holomorphic c=24 VOAs = the 71 N-shapes = the complete Fermat forbidden zone.  
Telperion IS the image of this map projected onto the CD tower.

**Three spaces, one tree:**
- **A**: Spherical Sedenion Space — latitude rings = CD levels, paths = great circle arcs
- **B**: Consecutive Euclidean planes — each level a flat cross-section, stacked vertically
- **C**: Fano Tower — 63 Fano heptagons (1+2+4+8+16+32) recording the sub-algebra structure

**Fractal boundary in all three:**  
- Space A: luminosity gradient on each latitude ring, modulated by prime density at that N-shape  
- Space B: point clouds around each quadrant node, density = prime count at that level and N-shape  
- Space C: heptagon luminosity = number of primes that pass through that Fano plane

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'engine'))
from telperion_engine import telperion_data, export_blender_json

N    = 1000
data = telperion_data(N)

print(f'Telperion dataset (N={N}):')
print(f'  Leaves (primes): {data["n_primes"]}')
print(f'  Monster gap:     {data["monster_gap"]}')
print(f'  Moonshine primes: {data["moonshine_primes"]}')

In [ ]:
# N-shape summary
ns = data['nshape_summary']
print(f'N-shape stratification of {data["n_primes"]} prime leaves:')
print(f'  {"ns":>4}  {"Count":>6}  {"Frac%":>7}  {"VOA source":<25}  Silver?')
print('-' * 65)
for k in range(16):
    e = ns[k]
    silver = '*** SILVER (Monster gap)' if e['is_gap'] else ''
    print(f'  e{k:2d}  {e["count"]:6d}  {e["fraction"]*100:7.3f}%  {e["voa_type"]:<25}  {silver}')

## Telperion geometry — all three spaces

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import numpy as np, math
from telperion_engine import prime_sieve, prime_tower_path, full_tower, NIEMEIER_GAP, MOONSHINE_PRIMES

primes = data['primes']
paths  = [prime_tower_path(p) for p in primes[:200]]  # first 200 for clarity

def prime_color(p):
    ns = p % 16
    if ns in NIEMEIER_GAP:  return 'silver'
    if ns in {3,5,7,9,13}:  return '#336699'   # Niemeier odd
    return '#aaaaaa'                             # even / Leech

fig = plt.figure(figsize=(18, 6))

# ── Space A: sphere ──────────────────────────────────────────────────────────
ax1 = fig.add_subplot(131, projection='3d')
for path in paths:
    col = prime_color(path['p'])
    xs  = [lv['sph']['x'] for lv in path['levels']]
    ys  = [lv['sph']['y'] for lv in path['levels']]
    zs  = [lv['sph']['z'] for lv in path['levels']]
    ax1.plot(xs, ys, zs, '-', color=col, linewidth=0.4, alpha=0.6)
ax1.set_title('Space A: Spherical Sedenion Space', fontsize=9)
ax1.set_axis_off()

# ── Space B: plane stack ──────────────────────────────────────────────────────
ax2 = fig.add_subplot(132)
tower = full_tower()
for lv in tower:
    ax2.axhline(lv['sigma']*4, color='#cccccc', linewidth=0.5)
    ax2.text(3.2, lv['sigma']*4, lv['name'], fontsize=6, va='center')
for path in paths:
    col = prime_color(path['p'])
    xs  = [lv['plane']['x'] for lv in path['levels']]
    zs  = [lv['plane']['z'] for lv in path['levels']]
    ax2.plot(xs, zs, '-', color=col, linewidth=0.3, alpha=0.5)
ax2.set_xlim(-3.5, 3.5)
ax2.set_xlabel('x')
ax2.set_ylabel('z (σ × 4)')
ax2.set_title('Space B: Consecutive Euclidean Planes', fontsize=9)

# ── Space C: Fano tower heat map ──────────────────────────────────────────────
ax3 = fig.add_subplot(133)

# Count primes per Fano plane at each level
fano_counts = {}  # (k, fi) -> count
for path in [prime_tower_path(p) for p in primes]:
    for lv in path['levels']:
        if lv['fano_idx'] is not None:
            key = (lv['k'], lv['fano_idx'])
            fano_counts[key] = fano_counts.get(key, 0) + 1

from telperion_engine import fano_tower_layout
ft = fano_tower_layout()
for row in ft:
    k   = row['k']
    nf  = row['n_fano']
    if nf == 0:
        continue
    row_w = max((nf-1)*1.5, 0.5)
    for pl in row['planes']:
        fi  = pl['fi']
        x   = -row_w/2 + fi*1.5 if nf > 1 else 0
        z   = row['sigma'] * 4
        cnt = fano_counts.get((k, fi), 0)
        max_cnt = max((fano_counts.get((k, f), 0) for f in range(nf)), default=1)
        brightness = cnt / max_cnt if max_cnt > 0 else 0
        col = 'silver' if pl['covers_gap'] else plt.cm.Blues(0.3 + 0.7*brightness)
        ax3.scatter([x], [z], s=max(20, brightness*80), color=col, alpha=0.9, zorder=3)

for row in ft:
    ax3.axhline(row['sigma']*4, color='#eeeeee', linewidth=0.3, zorder=1)
ax3.set_xlabel('Fano plane x position')
ax3.set_ylabel('z (σ × 4)')
ax3.set_title('Space C: Fano Tower (silver = Monster gap)', fontsize=9)

plt.suptitle(f'Telperion — {len(primes)} prime leaves, three coordinate spaces (N≤{N})', y=1.01)
plt.tight_layout()
plt.savefig('04_telperion_three_spaces.png', dpi=150)
plt.show()

## Fractal boundary — all three spaces

In [ ]:
from telperion_engine import fractal_boundary_data, prime_density_at_level

fb = fractal_boundary_data(primes)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={'polar': True})

for ax, k_val in zip(axes, [0, 4, 8]):
    lv   = fb['level_boundary'][k_val]
    phis = [np.radians(ns * 22.5) for ns in range(16)]
    rs   = [lv['nshape_density'][ns]['radius'] for ns in range(16)]
    cols = ['silver' if ns in NIEMEIER_GAP else '#334466' for ns in range(16)]

    # Close the polygon
    phis_closed = phis + [phis[0]]
    rs_closed   = rs   + [rs[0]]

    ax.fill(phis_closed, rs_closed, alpha=0.15, color='#334466')
    ax.plot(phis_closed, rs_closed, color='#334466', linewidth=0.8)
    for phi, r, col in zip(phis, rs, cols):
        ax.scatter([phi], [r], color=col, s=30, zorder=5)

    tower_lv = full_tower()[k_val]
    ax.set_title(f'k={k_val} {tower_lv["name"]} σ={tower_lv["sigma"]:+.2f}\n'
                 f'{"ZD equator" if k_val==4 else ("Leaf level" if k_val==0 else "Root")}',
                 pad=15, fontsize=9)
    ax.set_rticks([])

plt.suptitle('Fractal boundary contour: prime density by N-shape at k=0, k=4, k=8', y=1.02)
plt.tight_layout()
plt.savefig('04_fractal_boundary_contours.png', dpi=150)
plt.show()

## Export data for Blender

In [ ]:
json_path = export_blender_json(N=1000)
print(f'Blender data written to: {json_path}')
import json, os
size = os.path.getsize(json_path)
print(f'File size: {size/1024:.1f} KB')

## Summary: the proof chain

```
Telperion = Zero Tree = Un-Extinctable Bulk

Primes (leaves at k=0):
    No non-trivial factorization
    → Cannot be expressed as ZD pair at any level
    → Survive all 9 CD levels
    → Reach T_256 root intact

Composites (n = a × b):
    Have non-trivial factors
    → At k=4 (𝕊, dim=16): factors expressible as ZD pair
    → |ab| = 0 despite |a|=|b|=1
    → Fall off the tree at k=4

Monster gap primes (p ≡ 1,11,15 mod 16):
    Survive all levels AND
    Occupy N-shapes unreachable by any Niemeier A/D/E root system
    → The silver leaves of Telperion
    → What the Monster Group exists to account for

Fermat's Nightmare (FLT n≥3):
    SAME algebraic fact as ZD at dim=16
    Two languages for one identity
    = The mechanism that makes Telperion un-extinctable

71 VOAs = 71 N-shapes = Telperion's complete stratification
```